In [2]:
from vllm import LLM, SamplingParams

# Load the AI model on the MI300 GPU
# This takes about a minute - you'll see loading messages
model_name = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Loading {model_name} on AMD MI300 GPU...")
llm = LLM(model=model_name)
print("✓ Model loaded and ready!")

Loading Qwen/Qwen3-4B-Instruct-2507 on AMD MI300 GPU...
INFO 11-03 10:27:57 [api_utils.py:273] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}
INFO 11-03 10:27:57 [model.py:623] Resolved architecture: Qwen3ForCausalLM
INFO 11-03 10:27:57 [model.py:1788] Using max model len 262144
INFO 11-03 10:27:57 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=2335) INFO 11-03 10:27:59 [core.py:116] Initializing a V1 LLM engine (v0.26.0) with config: model='Qwen/Qwen3-4B-Instruct-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Instruct-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=262144, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_back

(EngineCore pid=2335) Process EngineCore:
(EngineCore pid=2335) Traceback (most recent call last):
(EngineCore pid=2335)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=2335)     self.run()
(EngineCore pid=2335)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=2335)     self._target(*self._args, **self._kwargs)
(EngineCore pid=2335)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1334, in run_engine_core
(EngineCore pid=2335)     raise e
(EngineCore pid=2335)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1299, in run_engine_core
(EngineCore pid=2335)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=2335)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=2335)   File "/usr/local/lib/python3.12/dist-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
(Engin

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {'EngineCore': 1}

In [ ]:
# Store our conversation history
conversation = []

# Set a reasonable response length
sampling_params = SamplingParams(max_tokens=200)

# Chat loop
while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break
    
    # Add user message to conversation
    conversation.append({"role": "user", "content": user_input})
    
    # This is INFERENCE - send to the model and get a response!
    outputs = llm.chat(conversation, sampling_params)
    bot_message = outputs[0].outputs[0].text
    
    # Add bot's response to conversation so it remembers context
    conversation.append({"role": "assistant", "content": bot_message})
    
    print(f"Bot: {bot_message}\n")

In [ ]:
# Create a conversation with a system prompt
shakespeare_conversation = [
    {
        "role": "system",  # This is the special system message
        "content": "You are a chatbot that speaks like William Shakespeare. Use Elizabethan English and poetic language. Be helpful but stay in character!"
    }
]

# Set a reasonable response length
sampling_params = SamplingParams(max_tokens=200)

# Chat loop
while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit"]:
        print("Farewell!")
        break
    
    shakespeare_conversation.append({"role": "user", "content": user_input})
    
    outputs = llm.chat(shakespeare_conversation, sampling_params)
    bot_message = outputs[0].outputs[0].text
    
    shakespeare_conversation.append({"role": "assistant", "content": bot_message})
    
    print(f"Shakespeare Bot: {bot_message}\n")

In [ ]:
# Create your own custom chatbot!
my_conversation = [
    {
        "role": "system",
        "content": "You are a friendly pirate. Talk like a pirate!"  # <-- Change this!
    }
]

# Test it with a quick message
my_conversation.append({"role": "user", "content": "Hello! Tell me about yourself."})

sampling_params = SamplingParams(max_tokens=200)
outputs = llm.chat(my_conversation, sampling_params)

print(outputs[0].outputs[0].text)

In [ ]:
# Experiment with different parameter combinations!
# Try changing these values and see how the response changes

sampling_params = SamplingParams(
    temperature=0.7,    # Try: 0.2 (safe), 0.7 (balanced), or 0.9 (creative)
    max_tokens=200,     # Try: 50 (short), 200 (medium), or 300 (long)
    top_p=0.9          # Try: 0.5 (more focused), 0.9 (balanced), or 1.0 (more diverse)
)

outputs = llm.chat(
    [{"role": "user", "content": "What's the best way to learn AI?"}],
    sampling_params
)

print(outputs[0].outputs[0].text)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create widgets
message_box = widgets.Textarea(
    placeholder='Type your message here...',
    description='Message:',
    layout=widgets.Layout(width='100%', height='80px')
)

temperature_slider = widgets.FloatSlider(
    value=0.7,
    min=0.1,
    max=1.0,
    step=0.1,
    description='Temperature:',
    style={'description_width': '150px'}
)

max_tokens_slider = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=10,
    description='Max Tokens:',
    style={'description_width': '150px'}
)

top_p_slider = widgets.FloatSlider(
    value=0.9,
    min=0.1,
    max=1.0,
    step=0.1,
    description='Top P:',
    style={'description_width': '150px'}
)

send_button = widgets.Button(
    description='Send',
    button_style='success',
    icon='paper-plane'
)

output_area = widgets.Output()

# Chat function
def on_send_clicked(b):
    with output_area:
        clear_output()
        message = message_box.value
        if message.strip():
            print(f"You: {message}\n")
            
            sampling_params = SamplingParams(
                temperature=temperature_slider.value,
                max_tokens=int(max_tokens_slider.value),
                top_p=top_p_slider.value
            )
            
            outputs = llm.chat([{"role": "user", "content": message}], sampling_params)
            response = outputs[0].outputs[0].text
            
            print(f"Bot: {response}")

send_button.on_click(on_send_clicked)

# Display the interface
display(widgets.HTML("<h3>AI Chatbot on AMD MI300</h3>"))
display(widgets.HTML("<p>Chat with AI running on AMD hardware!</p>"))
display(message_box)
display(temperature_slider)
display(max_tokens_slider)
display(top_p_slider)
display(send_button)
display(output_area)